# WP2 EM-DRATS (Environment for the Monitoring of Demographics, Retention, Affect, Tolerability, and Stratification)

requires:
Drop out questions_XXX.csv
wp2_post_session_4_XXX.csv
wp2_post_sessions_1-3_XXX.csv
wp2_pre_screen_XXX.csv
wp2_pre_session_1_XXX.csv
wp2_pre_sessions_2-4_XXX.csv
wp2_sms_day1,3,5_XXX.csv
wp2_sms_post_XXX.csv

creates:
wp2_assignments.csv
stratified & blinded sequence name per new participant ID
monitoring of wellbeing & tolerability
monitoring of demographic data, exclusion & retention rate

script 1: sign-up, demographic, retention (not for researcher use -- PI use only)
- run after running script 2 as it requires wp2_assignments

script 2: main blind stratification script (for researcher use)
- requires pre-screen CSV and creates list of blinded sequence names per new successful bookings (calendly_event_booked)
- below it is a stratification balance check, though it exposes number of strata per condition, so may unblind (not for researcher use -- PI use only)
- below the stratification balance check is the fixed seed (=42) through which our 5 control and 5 intervention sequences were renamed -- no need to ever run this again

script 3: main longitudinal wellbeing monitoring per condition/measure (not for researcher use -- PI use only)
- requires all 7 wellbeing CSVs
- plots per measure per condition with "worsening" part_ids outlined in summary tables below each measure plot
- used to see when clinicians should reach out to specific participants (touch base with researcher to link part_id to email)

script 4: main tolerability monitoring per condition (for any use)
- requires all post-session CSVs to make sure that session tolerability never exceeds our specified threshold

# WP2 Sign-Up & Retention Data Monitoring
*   requires wp2_pre_screen_XXX.CSV
*   requires drop out questions_XXX.CSV
*   requires wp2_assignments.CSV from stratification script
*   monitors passed/excluded as well as retention rate








In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

#####
# Setup
#####
DATA_DIR = Path('/content')

def find_csv(substring: str) -> Path:
    """Find exactly one CSV in DATA_DIR whose filename contains substring."""
    matches = list(DATA_DIR.glob(f'*{substring}*.csv'))
    if not matches:
        raise FileNotFoundError(f"No file matching '{substring}' in {DATA_DIR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple files match '{substring}': {matches}")
    return matches[0]

#####
# 0) Load assignments
#####
assign = pd.read_csv(find_csv('wp2_assignments'), dtype=str)

# Pools of sequence codes for inferring arm later
INTERVENTION_SEQS = assign.loc[assign['condition']=='intervention', 'sequence_name'].unique().tolist()
CONTROL_SEQS      = assign.loc[assign['condition']=='control',      'sequence_name'].unique().tolist()

#####
# 0.1) Define participant ID categories with comments
#####
# IDs who never attended Session 1 (booked but no-shows or withdrawn before arrival):
NEVER_ATTENDED_IDS = {
    "11274", # unable to make first session; excluding preliminarily
    "11380", # successful reschedule, ignore first iteration in pre-screening file
    "11266", # cancelled without rescheduling
    "11198", # reserved for manual scheduling (no-show)
    "11382", # withdrew before Session 1, data deletion requested
    "11290", # missed Session 1 and never rescheduled
    "11302", # missed Session 1; pending reschedule
    "11357", # no-show for Session 1
    "11379", # no-show for Session 1
    "11254", # manual scheduling issue; effectively no-show
    "11298", # did not show up for Session 1
    "11187", # no-show for Session 1
    "11296",  # ineligible: not depressed
    "11316", # Session 1 no-show
    "11388", # Session 1 no-show
    "11286", # not in calendar; manual scheduling?
    "11182", # Session 1 no-show
    "11226", # not in calendar, assuming manual scheduling
    "11341", # cancelled before Session 1
    "11121", # Session 1 no-show
    "11245", # Session 1 no-show
    "11215", # Session 1 no-show
  # "11134" # alleged rescheduling ???
    "11180", # no-show
    "11160", # Session 1 no-show, reach out to reschedule ???
    "11206", # Session 1 no-show
    "11199", # Session 1 no-show
    "11323", # Session 1 cancellation, reach out to shift +1 week (start 24/10/2025 ???)
    "11160", # Session 1 no-show
    "11184", # Session 1 no-show
    "11116", # Session 1 no-show
    "21131", # Sesssion 1 no-show
    "21246", # Session 1 no-show
    "21278" # Session 1 no-show.
    "11331",
    "21261",
    "21294"
}

# IDs who attended Session 1 but later dropped out (post-S1 drop-outs):
DROPOUT_AFTER_S1_IDS = {
    "11309", # dropout due to life events after Session 1
    "11162", # dropped out due to life events after Session 2
    "11152",  # too sick to make Session 2; will need to re-signup once feeling better
    "11213", # dropped out halfway through Session 1, said it was scary
    "11186", # dropped out after Session 1, did not like the experiment
    "11195", # drop-out
    "11383", # dropped after Session 3 due to moving, etc.
    "11215", # dropped out after Session 1
    "21399" # dropped out after Session 1

}

# IDs who dropped out after the taster session (not counted as post-S1 drop-outs):
TASTER_DROPOUT_IDS = {
    "11153"  # dropped out after taster session, exclude from S1 retention
}

# Derive true no-shows by removing any overlap
NOT_ATTENDED_IDS = NEVER_ATTENDED_IDS - DROPOUT_AFTER_S1_IDS - TASTER_DROPOUT_IDS
assert NOT_ATTENDED_IDS.isdisjoint(DROPOUT_AFTER_S1_IDS.union(TASTER_DROPOUT_IDS)), "ID sets overlap!"

# Sequence mappings for post-S1 drop-outs to infer condition
DROP_SEQ = {
    "11309": "WP2-6E363F", #
    "11162": "WP2-6E363F", #
    "11152": "WP2-3EB089",  #
    "11213": "WP2-7F6D87",
    "11186": "WP2-641C50",
    "11195": "WP2-C4546D",
    "11383": "WP2-641C50",
    "11180": "WP2-6E363F",
    "11215": "WP2-C4546D",
    "21399": "WP2-FC4114"
}

#####
# 1) Load & summarize pre-screening
#####
df_pre = pd.read_csv(find_csv('wp2_pre_screen_'), dtype=str)
df_pre.columns = df_pre.columns.str.strip()

if 'part_id' not in df_pre.columns:
    df_pre.rename(columns={df_pre.columns[0]: 'part_id'}, inplace=True)
df_pre['part_id'] = df_pre['part_id'].fillna('').astype(str)
# Remove phantom import rows
df_pre = df_pre[~df_pre['part_id'].str.contains('ImportId', na=False)].copy()

# Keep only completed screeners or safety-outs
safety_cols = [f'safety_{i}' for i in range(1, 8)]
df_pre = df_pre[
    df_pre['incl_dem_light'].notna() |
    df_pre[safety_cols].notna().any(axis=1)
].copy()

# Exclusion summary
exc_counts = df_pre['excluded'].value_counts()
summary_excl = pd.DataFrame({
    'Category': [
        '# Passed pre-screen',
        '# Excluded for safety',
        '# Excluded for demographics',
        '# Excluded for PHQ-9'
    ],
    'Count': [
        exc_counts.get('FALSE', 0),
        exc_counts.get('safety_screening', 0),
        exc_counts.get('demographics', 0),
        exc_counts.get('phq_9', 0)
    ]
})
print("=== Pre-screen Exclusion Summary ===")
display(summary_excl)

# Demographics for passed pre-screen
passed_df = df_pre[df_pre['excluded'] == 'FALSE'].copy()
ages = pd.to_numeric(passed_df['incl_dem_age'], errors='coerce')
print(f"\n=== Passed Participants Demographics (n={len(passed_df)}) ===")
print(f"Age    Mean±SD: {ages.mean():.1f}±{ages.std():.1f}")
sex_counts = passed_df['incl_dem_sex'].fillna('Missing').str.capitalize().value_counts().rename_axis('Sex').reset_index(name='Count')
print("\nSex distribution:")
display(sex_counts)
if 'incl_dem_gender' in passed_df.columns:
    gender_counts = passed_df['incl_dem_gender'].fillna('Missing').str.capitalize().value_counts().rename_axis('Gender').reset_index(name='Count')
    print("\nGender distribution:")
    display(gender_counts)

#####
# 2) Load & clean Session 1 sign-ups
#####
df_s1 = pd.read_csv(find_csv('wp2_pre_session_1'), dtype=str, skiprows=1)
df_s1.columns = df_s1.columns.str.strip()
if 'part_id' not in df_s1.columns:
    df_s1.rename(columns={df_s1.columns[0]: 'part_id'}, inplace=True)
df_s1['part_id'] = df_s1['part_id'].fillna('').astype(str)
# Remove phantom rows
df_s1 = df_s1[~df_s1['part_id'].str.startswith('{"ImportId', na=False)].copy()

#####
# 3) Summarize Qualtrics drop-out reasons (filtered to true post-S1 drop-outs)
#####
df_drop = pd.read_csv(find_csv('Drop out questions_'), dtype=str)
df_drop.columns = df_drop.columns.str.strip()
if 'part_id' not in df_drop.columns:
    df_drop.rename(columns={df_drop.columns[0]: 'part_id'}, inplace=True)

# Merge assignment condition
df_drop = df_drop.merge(assign[['part_id','condition']], on='part_id', how='left')

# Map free-text reasons to categories
def map_reason(text):
    if pd.isna(text): return 'unknown'
    t = text.lower()
    if 'uncomfortable' in t: return 'tolerability'
    if 'too difficult' in t: return 'scheduling'
    if text.startswith('I did not enjoy'): return 'other'
    return 'unknown'

df_drop['reason_cat'] = df_drop['drop_out_why'].apply(map_reason)
# Filter to only post-S1 drop-out IDs
df_drop = df_drop[df_drop['part_id'].isin(DROPOUT_AFTER_S1_IDS)]

# Unique post-S1 dropouts
post1_drop_unique = df_drop[['part_id','condition','reason_cat']].drop_duplicates('part_id')
n_dropped_overall = len(DROPOUT_AFTER_S1_IDS)
print(f"\n=== Drop-out Reasons (Overall: {n_dropped_overall} dropped post- Session 1) ===")
display(pd.DataFrame({
    'Reason': ['Tolerability','Scheduling','Other'],
    'Count': [
        sum(post1_drop_unique['reason_cat']=='tolerability'),
        sum(post1_drop_unique['reason_cat']=='scheduling'),
        sum(post1_drop_unique['reason_cat']=='other')
    ]
}))

#####
# 4) Narrative & retention summary
#####
# 4.1) Sign-ups among passed
signed_ids = set(passed_df.loc[
    passed_df['calendly_event_booked'].fillna('').str.strip().str.lower().isin(['true','none_available']),
    'part_id'
])

# 4.2) Attended Session 1 (minus true no-shows)
attended_ids = (signed_ids & set(df_s1['part_id'])) - NOT_ATTENDED_IDS
attended_df = passed_df[passed_df['part_id'].isin(attended_ids)]
print(f"\n=== Attended Participants Demographics (n={len(attended_df)}) ===")
ages_att = pd.to_numeric(attended_df['incl_dem_age'], errors='coerce')
print(f"Age    Mean±SD: {ages_att.mean():.1f}±{ages_att.std():.1f}")
sex_att = attended_df['incl_dem_sex'].fillna('Missing').str.capitalize().value_counts().rename_axis('Sex').reset_index(name='Count')
print("\nSex distribution:")
display(sex_att)
if 'incl_dem_gender' in attended_df.columns:
    gender_att = attended_df['incl_dem_gender'].fillna('Missing').str.capitalize().value_counts().rename_axis('Gender').reset_index(name='Count')
    print("\nGender distribution:")
    display(gender_att)

# 4.3) Manual post-S1 dropouts
md_ids = DROPOUT_AFTER_S1_IDS & attended_ids
manual_drop_n = len(md_ids)

# 4.4) Retention
attended_n = len(attended_ids)
retained_n = attended_n - manual_drop_n
post_ret   = retained_n / attended_n if attended_n else 0

# 4.5) Summary
passed_n    = len(passed_df)
int_passed  = assign[(assign['condition']=='intervention') & (assign['part_id'].isin(passed_df['part_id']))]['part_id'].nunique()
ctrl_passed = assign[(assign['condition']=='control')      & (assign['part_id'].isin(passed_df['part_id']))]['part_id'].nunique()
int_signed  = sum((assign['part_id'].isin(signed_ids))   & (assign['condition']=='intervention'))
ctrl_signed = sum((assign['part_id'].isin(signed_ids))   & (assign['condition']=='control'))
int_att     = sum((assign['part_id'].isin(attended_ids)) & (assign['condition']=='intervention'))
ctrl_att    = sum((assign['part_id'].isin(attended_ids)) & (assign['condition']=='control'))

print(f"""
1) Of the {passed_n} who passed the pre-screen, {len(signed_ids)} signed up for Session 1.
2) Of those {len(signed_ids)}, {len(attended_ids)} attended Session 1.
3) Of those {len(attended_ids)}, {manual_drop_n} dropped out → retention {retained_n}/{len(attended_ids)} ({post_ret:.1%}).

Sign-ups by arm:
  • Intervention: {int_signed}/{int_passed} ({int_signed/int_passed:.1%})
  • Control:      {ctrl_signed}/{ctrl_passed} ({ctrl_signed/ctrl_passed:.1%})

Attendance by arm:
  • Intervention: {int_att}/{int_signed} ({int_att/int_signed:.1%})
  • Control:      {ctrl_att}/{ctrl_signed} ({ctrl_att/ctrl_signed:.1%})

Post-Session 1 drop-outs by arm:
  • Intervention: {sum(1 for pid in md_ids if DROP_SEQ.get(pid) in INTERVENTION_SEQS)}/{int_att} → retention {(int_att - sum(1 for pid in md_ids if DROP_SEQ.get(pid) in INTERVENTION_SEQS))/int_att:.1%}
  • Control:      {sum(1 for pid in md_ids if DROP_SEQ.get(pid) in CONTROL_SEQS)}/{ctrl_att} → retention {(ctrl_att - sum(1 for pid in md_ids if DROP_SEQ.get(pid) in CONTROL_SEQS))/ctrl_att:.1%}
"""
)

# Participant Stratification

block size = 4 & strata = 4 (low/high PHQ-9 * yes/no pharmacological intervention)


*   requires wp2_pre_screen_XXX.CSV
*   requires wp2_assignments.CSV once script has been ran more than once to ensure participants are equally allocated and informed by previous randomizations -- MAKE SURE TO SAVE AND DATE THE LAST ITERATION!!! (then, when reuploading, remove data marker [just "wp2_assignments"])

In [ ]:
import pandas as pd
import random
import hashlib
from pathlib import Path

#####
# Parameters
#####
DATA_DIR   = Path('/content')
OUTPUT_CSV = Path('wp2_assignments.csv')
BLOCK_SIZE = 4
SEED       = 42  # None for non-reproducible

#####
# 0) Define participant categories
#####
# IDs who never attended Session 1 (booked but no-shows or withdrew before arrival):
NEVER_ATTENDED_IDS = {
    "11274", # unable to make first session; excluding preliminarily
    "11380", # successful reschedule, ignore first iteration
    "11266", # cancelled without rescheduling
    "11198", # reserved for manual scheduling (no-show)
    "11382", # withdrew before S1, data deletion requested
    "11290", # missed S1 and never rescheduled
    #"11302", # missed S1; pending reschedule
    "11357", # no-show S1
    "11379", # no-show S1
    "11393", # missed S1 after second reschedule
    "11316", # missed S1 and never rescheduled
    "11254", # manual scheduling issue; effectively no-show
    "11298", # did not show up for S1
    "11187", # no-show for S1
    "11296",  # ineligible: not depressed
    "11388", # no-show for Session 1
    "11286", # not in calendar; manual scheduling?
    "11182", # Session 1 no-show
    "11226", # not in calendar, assuming manual scheduling
    "11341", # cancelled before Session 1
    "11121", # Session 1 no-show
    "11245", # Session 1 no-show
    "11215", # Session 1 no-show
  # "11134" # alleged rescheduling ???
    "11180", # no-show
    "11160", # Session 1 no-show, reach out to reschedule ???
    "11206", # Session 1 no-show
    "11199", # Session 1 no-show
    #"11323", # Session 1 cancellation, reach out to shift +1 week (start 24/10/2025 ???)
    "11160", # Session 1 no-show
    "11184", # Session 1 no-show
    "11116", # Session 1 no-show
    "21136", # no
    "21189", # no
    "21131", # Sesssion 1 no-show
    "21246", # Session 1 no-show
    "21278" # Session 1 no-show
}

# IDs who dropped out after the taster session
TASTER_DROPOUT_IDS = {"11153"}
# IDs who attended S1 but later dropped out (true post-S1 drop-outs)
DROPOUT_AFTER_S1_IDS = {"11309","11162","11152", "11213", "11186", "11195", "11383", "11215", "21399"}

# Combine all exclusions before randomization
EXCLUDE_IDS = NEVER_ATTENDED_IDS.union(TASTER_DROPOUT_IDS).union(DROPOUT_AFTER_S1_IDS)

#####
# 1) Utility to find exactly one CSV
#####
def find_csv(sub: str) -> Path:
    matches = list(DATA_DIR.glob(f"*{sub}*.csv"))
    if not matches:
        raise FileNotFoundError(f"No file matching '{sub}' in {DATA_DIR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple files match '{sub}': {matches}")
    return matches[0]

#####
# 2) Generate blinded sequence pools
#####
def gen_sequences(condition: str, count: int = 5) -> list[str]:
    return [
        f"WP2-{hashlib.sha256(f'{SEED}-{condition}-{i}'.encode()).hexdigest().upper()[:6]}"
        for i in range(count)
    ]
INTERVENTION_SEQS = gen_sequences('intervention')
CONTROL_SEQS      = gen_sequences('control')

#####
# 3) Load & filter pre-screen
#####
df = pd.read_csv(find_csv('wp2_pre_screen_'), dtype=str)
df.columns    = df.columns.str.strip()
df['part_id'] = df['part_id'].str.strip()

# (a) only those who passed pre-screen
df = df[df['excluded'].str.strip().str.lower() == 'false'].copy()
# (b) only those who booked or none_available
df['cal_book'] = df['calendly_event_booked'].fillna('').str.strip().str.lower()
df = df[df['cal_book'].isin(['true','none_available'])].copy()
df.drop(columns=['cal_book'], inplace=True)
# (c) remove all exclusion categories before randomization
df = df[~df['part_id'].isin(EXCLUDE_IDS)].copy()

# define strata
df['phq9_sum'] = pd.to_numeric(df['phq9_sum'], errors='coerce')
def phq9_bin(x):
    if 5 <= x <= 16: return 'low'
    if 17 <= x <= 27: return 'high'
    return None
df['stratum'] = df['phq9_sum'].apply(phq9_bin).fillna('') + '_' + df['incl_dem_med'].astype(str).str.strip().str.lower()
df = df[df['stratum'].str.contains('_')].copy()
# preserve arrival order
df['_order'] = range(len(df))

# ==== AUDIT: who got filtered out, and why? ====
raw = pd.read_csv(find_csv('wp2_pre_screen_'), dtype=str)
raw['part_id'] = raw['part_id'].astype(str).str.strip()

# Normalize the key flags just like Section 3
raw['excluded_norm'] = raw['excluded'].astype(str).str.strip().str.lower()
raw['cal_book_norm'] = raw['calendly_event_booked'].astype(str).str.strip().str.lower()
raw['phq9_sum_num']  = pd.to_numeric(raw['phq9_sum'], errors='coerce')
raw['incl_dem_med_norm'] = raw['incl_dem_med'].astype(str).str.strip().str.lower()

def phq_bin(x):
    if pd.isna(x): return None
    if 5 <= x <= 16: return 'low'
    if 17 <= x <= 27: return 'high'
    return None

raw['phq9_cat'] = raw['phq9_sum_num'].apply(phq_bin)

# Final eligibility rule mirrors your Section 3
elig = (
    (raw['excluded_norm'] == 'false') &
    (raw['cal_book_norm'].isin(['true','none_available'])) &
    (~raw['part_id'].isin(EXCLUDE_IDS)) &
    (raw['phq9_cat'].notna()) &
    (raw['incl_dem_med_norm'].isin(['yes','no','true','false']))  # accept yes/no variants
)

raw['eligible_now'] = elig

# Load existing assignment ids (post-EXCLUDE_IDS)
if OUTPUT_CSV.exists():
    ex = pd.read_csv(OUTPUT_CSV, dtype=str)
    ex['part_id'] = ex['part_id'].str.strip()
    ex = ex[~ex['part_id'].isin(EXCLUDE_IDS)]
    existing_ids = set(ex['part_id'])
else:
    existing_ids = set()

raw['already_assigned'] = raw['part_id'].isin(existing_ids)

# Classify reason for not appearing as "newly assigned"
def reason_row(r):
    if not r['eligible_now']:
        reasons = []
        if r['excluded_norm'] != 'false': reasons.append('excluded!=false')
        if r['cal_book_norm'] not in ['true','none_available']: reasons.append('not_booked')
        if r['part_id'] in EXCLUDE_IDS: reasons.append('in_EXCLUDE_IDS')
        if pd.isna(r['phq9_cat']): reasons.append('phq9_invalid')
        if r['incl_dem_med_norm'] not in ['yes','no','true','false']: reasons.append('incl_dem_med_invalid')
        return ';'.join(reasons) or 'ineligible'
    if r['already_assigned']:
        return 'already_in_assignments'
    return 'NEW'  # would be newly assigned

raw['status_reason'] = raw.apply(reason_row, axis=1)

# Summary
print("\n=== AUDIT SUMMARY ===")
print(raw['status_reason'].value_counts())

print("\nIDs that WOULD be new if eligible & not already assigned:")
print(raw.loc[(raw['eligible_now']) & (~raw['already_assigned']), 'part_id'].tolist()[:20])

print("\nSample of filtered-out with reasons:")
print(raw.loc[~raw['eligible_now'], ['part_id','excluded_norm','cal_book_norm','phq9_sum','incl_dem_med','status_reason']].head(20))

#####
# 4) Load existing assignments & free slots
#####
if OUTPUT_CSV.exists():
    existing = pd.read_csv(OUTPUT_CSV, dtype=str)
    existing.columns    = existing.columns.str.strip()
    existing['part_id'] = existing['part_id'].str.strip()
    # free up slots for excluded IDs
    existing = existing[~existing['part_id'].isin(EXCLUDE_IDS)].copy()
else:
    existing = pd.DataFrame(columns=['part_id','stratum','condition','sequence_name'])

# Keep all columns we’ll need from existing; DO NOT recompute these for existing rows
assigned = existing[['part_id','stratum','condition','sequence_name']].copy()

#####
# 5) Merge to find new participants (merge ONLY on part_id!)
#####
merged = df[['part_id','stratum','_order']].merge(
    assigned.rename(columns={'stratum':'existing_stratum'}),
    on='part_id', how='left'
).reset_index(drop=True)

# If the participant already exists, keep their original condition & sequence
merged['stratum_final'] = merged.apply(
    lambda r: r['existing_stratum'] if pd.notna(r['existing_stratum']) else r['stratum'], axis=1
)

# A participant is "new" iff they have no existing condition
merged['is_new'] = merged['condition'].isna()

#####
# 6) Block randomization per stratum (only for NEW rows)
#####
# Count how many participants already assigned per (their original) stratum.
# This preserves past assignments and avoids re-stratifying them.
existing_counts = assigned.groupby('stratum').size() if not assigned.empty else pd.Series(dtype=int)

# Compute an index within each stratum for new arrivals, offset by existing counts
mask_new = merged['is_new']
start_counts = merged.loc[mask_new, 'stratum'].map(existing_counts).fillna(0).astype(int)
new_cumcnt   = merged.loc[mask_new].groupby('stratum').cumcount()

merged.loc[mask_new, 'stratum_idx'] = start_counts.values + new_cumcnt.values
merged['stratum_idx'] = merged['stratum_idx'].fillna(0).astype(int)

def assign_block_stratum(stratum: str, idx: int) -> str:
    block_num = idx // BLOCK_SIZE
    seed_int  = None if SEED is None else int(hashlib.sha256(f"{SEED}-{stratum}-{block_num}".encode()).hexdigest(), 16)
    rng = random.Random(seed_int)
    block = ['intervention']*(BLOCK_SIZE//2) + ['control']*(BLOCK_SIZE//2)
    rng.shuffle(block)
    return block[idx % BLOCK_SIZE]

# Only assign conditions for new rows; keep existing conditions untouched
merged.loc[mask_new, 'condition'] = merged.loc[mask_new].apply(
    lambda r: assign_block_stratum(r['stratum'], int(r['stratum_idx'])), axis=1
)

#####
# 7) Assign sequence codes (ONLY for new rows; offset by existing counts per condition)
#####
merged = merged.sort_values('_order').reset_index(drop=True)

# How many sequences already used per condition?
existing_cond_counts = assigned.groupby('condition').size().to_dict() if not assigned.empty else {}
offset_I = int(existing_cond_counts.get('intervention', 0))
offset_C = int(existing_cond_counts.get('control', 0))

# Give each NEW row a per-condition sequence index starting after existing ones
mask_I_new = mask_new & (merged['condition'] == 'intervention')
mask_C_new = mask_new & (merged['condition'] == 'control')

merged.loc[mask_I_new, 'seq_idx'] = offset_I + merged.loc[mask_I_new].groupby('condition').cumcount()
merged.loc[mask_C_new, 'seq_idx'] = offset_C + merged.loc[mask_C_new].groupby('condition').cumcount()
merged['seq_idx'] = merged['seq_idx'].fillna(-1).astype(int)  # -1 means "existing row"

def seq_name_from_idx(cond: str, idx: int) -> str:
    if cond == 'intervention':
        return INTERVENTION_SEQS[idx % len(INTERVENTION_SEQS)]
    else:
        return CONTROL_SEQS[idx % len(CONTROL_SEQS)]

# Only set sequence_name for new rows; keep existing sequence_name as-is
merged.loc[mask_new, 'sequence_name'] = merged.loc[mask_new].apply(
    lambda r: seq_name_from_idx(r['condition'], r['seq_idx']), axis=1
)

#####
# 8) Save & report (preserve existing assignments exactly)
#####
full = merged[['part_id','stratum','condition','sequence_name']].copy()
full.to_csv(OUTPUT_CSV, index=False)

print("Newly assigned participants:")
print(merged.loc[merged['is_new'], ['part_id','sequence_name']])

Blinded Sequence Name Generator, 5/5 Intervention/Control with a Fixed Seed
*   the same seed is used for the stratification cells above for researchers to remain blinded

In [ ]:
#####
# 7) Duplicate template files for each sequence
#####
import shutil
# Define your template text files (adjust paths if needed)
CONTROL_TEMPLATE       = DATA_DIR / 'z_control_adj2.txt'
INTERVENTION_TEMPLATE = DATA_DIR / 'z_intervention_adj2.txt'

# Ensure templates exist
if not CONTROL_TEMPLATE.exists():
    raise FileNotFoundError(f"Control template not found at {CONTROL_TEMPLATE}")
if not INTERVENTION_TEMPLATE.exists():
    raise FileNotFoundError(f"Intervention template not found at {INTERVENTION_TEMPLATE}")

# Create copies named by sequence codes
for seq in CONTROL_SEQS:
    dest = DATA_DIR / f"{seq}.txt"
    shutil.copy(CONTROL_TEMPLATE, dest)

for seq in INTERVENTION_SEQS:
    dest = DATA_DIR / f"{seq}.txt"
    shutil.copy(INTERVENTION_TEMPLATE, dest)

print(f"Created {len(CONTROL_SEQS)} control and {len(INTERVENTION_SEQS)} intervention files.")

blinded debriefing

In [ ]:
from datetime import datetime

def generate_debrief(part_id: str, assignments_df: pd.DataFrame) -> str:
    """
    Given a part_id and the augmented assignments DataFrame,
    return your personalized debrief paragraph.
    """
    # look up the row
    row = assignments_df.loc[assignments_df['part_id'] == part_id]
    if row.empty:
        raise ValueError(f"No assignment found for part_id {part_id!r}")

    # pull & format condition and day
    condition = row['condition'].iloc[0].capitalize()
    day_of_week = datetime.now().strftime('%A')

    # fill the template
    return f"""Hello again and Happy {day_of_week}!

    Thank you for participating in our investigation into strobe light therapy as a treatment of depression.

    We wanted to let you know that you were in the {condition} condition.

    Attached (and printed below) is your debrief information. If you have any questions, please don't hesitate to get in touch.

    Depressive disorders are a leading contributor to the global health-related burden. There is
    great need for the development of cost-effective and sustainable therapeutic treatments
    that can complement existing treatment approaches for depression.

    One existing treatment, bright light therapy, has been shown to reduce symptoms of
    seasonal affective disorder (SAD) and non-seasonal depression. Stroboscopic light has
    been suggested to have enhanced therapeutic effects over bright light therapy. In a mouse
    model of depression, relatively short exposure to stroboscopic light stimulation was shown
    to reduce depression-like behaviours significantly better than the commonly used anti-depressant fluoxetine (Kim et al, 2016).
    In humans, stroboscopic light on closed eyes, at certain frequencies, typically gives rise to
    vivid visual experiences (e.g., colours, geometric patterns, movement, complex scenes), as
    well as, for some people, powerful emotional responses. Recently, we completed a large-scale public art-science experience, the Dreamachine, which enabled more than 40,000
    people to safely experience visual hallucinations induced by strobe light. Everyone
    participating in this event had a unique experience, despite being exposed to the same
    flickering white light. Almost all participants reported a new appreciation of the power of
    their own minds and brains to create their perceptual experiences. The vast majority also
    reported positive emotions, including many spontaneous reports of positive benefits for
    depression, anxiety, and related mental states.

    The goal of this study is to assess the feasibility and tolerability of a stroboscopic
    intervention for depressive symptoms, while controlling for the influence of expectation on
    the potential efficacy of the strobe sessions. To control for placebo effects - where people
    may report improvements simply because they believe they are receiving an effective
    treatment - this study included both an experimental and a control condition. Researchers
    were blinded to which condition participants were randomly assigned to. Participants in
    both groups completed the same questionnaires and four strobe light sessions. However,
    the control condition's strobe sessions were slightly altered so that they would cause
    minimal stroboscopic experiences.

    It is our hope that this study will deliver the critical data needed to enable further testing of
    stroboscopic light therapy for depression.

    Again, we stress that this study is not a clinical trial and is intended as an initial step in
    assessing the potential of stroboscopic therapy to reduce depressive symptoms.

    If you are concerned about your own emotional wellbeing or mental health, then here
    are some sources of support to consider:

    https://www.https://www.mind.org.uk/information-support/types-of-mental-health-problems/seasonal-affectivedisorder-sad/about-sad/

    If you are a student at the University of Sussex:

    Student Life Centre (University of Sussex)
    Website: https://www.sussex.ac.uk/studentlifecentre/
    Phone: 01273 876767 Email: studentlifecentre@sussex.ac.uk

    If you have any questions or concerns about this study, please get in touch with the lead
    researcher Dr. David Schwartzman (d.schwartzman@sussex.ac.uk).

    Thank you for your participation in this study.

    Dr. David Schwartzman - Postdoctoral Research Fellow
    The Sussex Centre for Consciousness Science
"""

# Enter part_id below to generate your email to copy and paste:
print(generate_debrief("11335", aug))

## Participant Wellbeing Monitoring

requires:
*   wp2_pre_session_1_XXX.CSV
*   wp2_sms_day1,3,5_XXX.CSV
*   wp2_pre_sessions_2-4_XXX.CSV
*   wp2_post_sessions_1-3_XXX.CSV
*   wp2_post_session_4_XXX.CSV
*   wp2_sms_post_XXX.CSV
*   wp2_assignments.CSV from the stratification script

This monitor will unblind researchers to part_id:condition allocation; please run the one below for a condition-blind part_id monitoring of wellbeing.


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

#####
# 0) SETTINGS
#####
DATA_DIR = Path('/content')

#####
# 1) UTILITY TO FIND EXACTLY ONE CSV
#####
def find_csv(substring: str) -> Path:
    matches = list(DATA_DIR.glob(f'*{substring}*.csv'))
    if not matches:
        raise FileNotFoundError(f"No file matching '{substring}' in {DATA_DIR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple files match '{substring}': {matches}")
    return matches[0]

#####
# 1b) TRY TO LOAD A CSV, BUT SKIP IF MISSING OR AMBIGUOUS
#####
def try_load(substring: str) -> pd.DataFrame | None:
    try:
        path = find_csv(substring)
    except (FileNotFoundError, ValueError) as e:
        print(f"Skipping '{substring}': {e}")
        return None
    print(f"Loaded '{substring}' from {path.name}")
    return pd.read_csv(path, dtype=str)

#####
# 2) LOAD ALL WELLBEING CSVS (skipping any that aren’t there yet)
#####
df_s1_pre     = try_load('pre_session_1')
df_s2to4_pre  = try_load('pre_sessions_2-4')
df_s2to4_post = try_load('post_sessions_1-3')
df_s4_post    = try_load('post_session_4')
df_sms        = try_load('sms_day1,3,5')

#  — new step: trim SMS part_ids —
if df_sms is not None:
    df_sms['part_id'] = (
        df_sms['part_id']
              .astype(str)
              .str.strip()
              .str[:5]
    )
    print("Trimmed df_sms.part_id to first 5 characters")

df_final      = try_load('sms_post')

# Collect only the ones that actually loaded
all_dfs = [df for df in (
    df_s1_pre,
    df_s2to4_pre,
    df_s2to4_post,
    df_s4_post,
    df_sms,
    df_final
) if df is not None]

if not all_dfs:
    raise RuntimeError("No data files found to process!")

#####
# 3) CLEAN BDI ITEMS (extract leading number)
#####
for df in all_dfs:
    bdi_raw = [c for c in df.columns if re.fullmatch(r"bdi_\d+(?:_1)?", c)]
    for col in bdi_raw:
        df[col] = (
            df[col].astype(str)
                  .str.extract(r'^(\d+)')[0]
                  .astype(float)
        )

#####
# 4) SUM SCALES
#####
def sum_scale(df, cols, new_col):
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
    df[new_col] = df[cols].sum(axis=1)

def get_scale_cols(df, prefix, count, allow_suffix=False):
    cols = []
    for i in range(1, count+1):
        main = f"{prefix}_{i}"
        alt  = f"{prefix}_{i}_1"
        if main in df.columns:
            cols.append(main)
        elif allow_suffix and alt in df.columns:
            cols.append(alt)
    return cols

for df in all_dfs:
    # BAI
    bai_cols = get_scale_cols(df, 'bai', 21)
    if bai_cols:
        sum_scale(df, bai_cols, 'bai_sum')
    # BDI
    bdi_cols = get_scale_cols(df, 'bdi', 21)
    if bdi_cols:
        sum_scale(df, bdi_cols, 'bdi_sum')
    # MADRS
    madrs_cols = get_scale_cols(df, 'madrs', 9, allow_suffix=True)
    if madrs_cols:
        sum_scale(df, madrs_cols, 'MADRS_S_sum')

#####
# 5) STANDARDIZE M3VAS
#####
def standardize_m3vas(df):
    for var in ('mood','pleasure','suicidal'):
        ch_cols  = [c for c in df.columns if re.fullmatch(f"m3vas_ch_{var}(?:_1)?", c)]
        raw_cols = [c for c in df.columns if re.fullmatch(f"m3vas_{var}(?:_1)?",    c)]
        if ch_cols:
            df[f"m3vas_{var}"] = pd.to_numeric(df[ch_cols[0]], errors='coerce') + 50
        elif raw_cols:
            df[f"m3vas_{var}"] = pd.to_numeric(df[raw_cols[0]], errors='coerce')
    # invert pleasure
    if 'm3vas_pleasure' in df.columns:
        df['m3vas_pleasure'] = 100 - df['m3vas_pleasure']
    # drop raw/input cols
    drop_pattern = re.compile(
        r"^(?:m3vas_ch_(?:mood|pleasure|suicidal)(?:_1)?|m3vas_(?:mood|pleasure|suicidal)_1)$"
    )
    to_drop = [c for c in df.columns if drop_pattern.match(c)]
    if to_drop:
        df.drop(columns=to_drop, inplace=True)

for df in all_dfs:
    standardize_m3vas(df)

#####
# 6) CLEAN FISBER
#####
for df in all_dfs:
    for col in ('fisber_1','fisber_2','fisber_3'):
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                       .str.extract(r'(\d+)$')[0]
                       .astype(float)
            )

#####
# 7) COERCE TYPES
#####
# session_n for sessions
for df in (df_s1_pre, df_s2to4_pre, df_s2to4_post, df_s4_post):
    if df is not None and 'session_n' in df.columns:
        df['session_n'] = pd.to_numeric(df['session_n'], errors='coerce').astype('Int64')

# session_n & sms_n for SMS
if df_sms is not None:
    for col in ('session_n','sms_n'):
        if col in df_sms.columns:
            df_sms[col] = pd.to_numeric(df_sms[col], errors='coerce').astype('Int64')

# numeric scales
numeric_cols = [
    'phq9_sum','spane_p','spane_n',
    'm3vas_mood','m3vas_pleasure','m3vas_suicidal',
    'fisber_1','fisber_2','fisber_3',
    'bai_sum','bdi_sum','MADRS_S_sum'
]
for df in all_dfs:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

#####
# 8) ENSURE EVERY DF HAS part_id
#####
for df in all_dfs:
    if 'part_id' not in df.columns:
        if 'ProlificID' in df.columns:
            df.rename(columns={'ProlificID':'part_id'}, inplace=True)
        else:
            df['part_id'] = df.index.astype(str)
    df['part_id'] = df['part_id'].astype(str).str.strip()

#####
# 9) TAG TIMEPOINTS
#####
if df_s1_pre    is not None: df_s1_pre    ['timepoint'] = 'session1_pre'
if df_s2to4_pre is not None: df_s2to4_pre ['timepoint'] = df_s2to4_pre.apply(lambda r: f"session{r.session_n}_pre",  axis=1)
if df_s2to4_post is not None: df_s2to4_post['timepoint'] = df_s2to4_post.apply(lambda r: f"session{r.session_n}_post", axis=1)
if df_s4_post   is not None: df_s4_post   ['timepoint'] = 'session4_post'
if df_sms       is not None: df_sms       ['timepoint'] = df_sms.apply(lambda r: f"SMS_{r.session_n}_{r.sms_n}", axis=1)
if df_final     is not None: df_final     ['timepoint'] = 'final_post'

#####
# 10) CONCATENATE
#####
df_long = pd.concat(all_dfs, ignore_index=True)

#####
# 11) DROP NON-NUMERIC part_id
#####
df_long = df_long[df_long['part_id'].str.match(r'^\d+$', na=False)]

#####
# 12) MERGE IN CONDITION (if assignments file was found)
#####
df_assign = try_load('wp2_assignments')
if df_assign is not None:
    df_long = df_long.merge(
        df_assign[['part_id','condition']],
        on='part_id',
        how='inner'
    )
else:
    print("No assignments file: skipping merge with 'condition' column")

#####
# 13) DEFINE & APPLY ORDERED CATEGORICAL FOR timepoint
#####
ordered_timepoints = [
    'session1_pre','session1_post',
    'SMS_1_1','SMS_1_2','SMS_1_3',
    'session2_pre','session2_post',
    'SMS_2_1','SMS_2_2','SMS_2_3',
    'session3_pre','session3_post',
    'SMS_3_1','SMS_3_2','SMS_3_3',
    'session4_pre','session4_post',
    'final_post'
]
df_long['timepoint'] = pd.Categorical(
    df_long['timepoint'],
    categories=ordered_timepoints,
    ordered=True
)

#####
# 14) PLOTTING + FLAGGING BY CONDITION
#####
bad_if_higher = {
    'phq9_sum',
    'spane_n',
    'fisber_1','fisber_2','fisber_3',
    'bai_sum','bdi_sum','MADRS_S_sum',
    'm3vas_suicidal'   # higher = more suicidal = worse
}

bad_if_lower = {
    'spane_p',
    'm3vas_mood',      # higher = better mood, so a drop is worsening
    'm3vas_pleasure'   # higher = better pleasure, so a drop is worsening
}

mcid_thresholds = {
    'phq9_sum': 5,
    'spane_p': 2,
    'spane_n': 2,
    'm3vas_mood': 12,
    'm3vas_pleasure': 12,
    'm3vas_suicidal': 12
}

#####
# ADJUST M3VAS SO 50 → 0 (“no change”), 0 → –50 (worse), 100 → +50 (better)
#####
def standardize_m3vas(df):
    to_drop = []
    for var in ('suicidal','mood','pleasure'):
        # find change‐score first, then raw‐score
        ch_col  = next((c for c in df.columns
                        if re.fullmatch(rf"m3vas_ch_{var}(?:_1)?", c)), None)
        raw_col = next((c for c in df.columns
                        if re.fullmatch(rf"m3vas_{var}(?:_1)?",    c)), None)

        if ch_col:
            vals = pd.to_numeric(df[ch_col], errors='coerce') - 50
            to_drop.append(ch_col)
        elif raw_col:
            vals = pd.to_numeric(df[raw_col], errors='coerce') - 50
            to_drop.append(raw_col)
        else:
            continue

        if var == 'suicidal':
            # leave as-is: + means more suicidal (worse), so improvement → down
            df[f"m3vas_{var}"] = vals
        else:
            # mood & pleasure: + means improvement, so flip sign
            df[f"m3vas_{var}"] = -vals

    # drop only the original raw/change columns
    df.drop(columns=to_drop, inplace=True, errors='ignore')

# apply to each dataframe
for df in all_dfs:
    standardize_m3vas(df)

def plot_and_flag_by_condition(measure):
    if measure not in df_long.columns:
        print(f"→ skipping '{measure}': not in df_long")
        return

    # For these measures, remove any session*_post timepoints
    df_plot = df_long.copy()
    if measure in ('phq9_sum','spane_p','spane_n'):
        df_plot = df_plot[~df_plot['timepoint'].str.match(r'^session\d+_post$')]

    for cond, dfc in df_plot.groupby('condition', dropna=False):
        print(f"\n── {measure}  |  Condition: {cond} ──")
        fig, ax = plt.subplots(figsize=(12,6))
        x_final = len(ordered_timepoints) - 1

        # Plot trajectories
        for pid, grp in dfc.groupby('part_id'):
            s = grp.dropna(subset=['timepoint', measure]).sort_values('timepoint')
            if s.empty:
                continue
            x = s['timepoint'].cat.codes
            y = s[measure].astype(float)
            ax.plot(x, y, linewidth=1, alpha=0.6)
            ax.text(x_final+0.2, y.iloc[-1], pid, fontsize=6, va='center')

        ax.set_xlim(-0.5, x_final+1.5)
        ax.set_xticks(range(len(ordered_timepoints)))
        ax.set_xticklabels(ordered_timepoints, rotation=45, ha='right')
        ax.set_ylabel(measure)
        ax.set_title(f"{measure} — {cond}")
        plt.tight_layout(); plt.show()

        # Flag first MCID‐level worsening events
        if measure in mcid_thresholds:
            th = mcid_thresholds[measure]
            breach_events = []
            for pid, grp_pid in dfc.groupby('part_id'):
                seq = (
                    grp_pid
                    .dropna(subset=['timepoint', measure])
                    .sort_values('timepoint')
                    .loc[:, ['part_id','timepoint', measure]]
                    .assign(change=lambda d: d[measure] - d[measure].iloc[0])
                )
                mask = seq['change'] >= th if measure in bad_if_higher else seq['change'] <= -th
                if mask.any():
                    breach_events.append(seq[mask].iloc[0])
            if breach_events:
                print(f"First MCID‐level worsening events (threshold ≥ {th}):")
                display(pd.DataFrame(breach_events))
            else:
                print(f"No participants worsened by at least the MCID of {th} for {measure}.")
        else:
            # fallback: ever‐worsened
            worsened = []
            for pid, grp_pid in dfc.groupby('part_id'):
                seq = grp_pid.dropna(subset=['timepoint', measure])\
                            .sort_values('timepoint')[measure].astype(float)
                if len(seq)<2: continue
                if ((measure in bad_if_higher and seq.iloc[-1]>seq.iloc[0]) or
                    (measure in bad_if_lower  and seq.iloc[-1]<seq.iloc[0])):
                    worsened.append(pid)
            if worsened:
                print("Worsened trajectories:")
                display(
                    dfc[dfc['part_id'].isin(worsened)]
                       [['part_id','timepoint',measure]]
                       .dropna(subset=[measure])
                       .sort_values(['part_id','timepoint'])
                       .reset_index(drop=True)
                )
            else:
                print("No worsening in this condition.")

# RUN PLOTS FOR ALL MEASURES
measures = [
    'phq9_sum','spane_p','spane_n',
    'm3vas_mood','m3vas_pleasure','m3vas_suicidal',
    'fisber_1','fisber_2','fisber_3',
    'bai_sum','bdi_sum','MADRS_S_sum'
]
#for m in measures:
#    plot_and_flag_by_condition(m)

## Condition-Blind Wellbeing Monitoring

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

#####
# 0) SETTINGS
#####
DATA_DIR = Path('/content')

#####
# 1) UTILITY TO FIND EXACTLY ONE CSV
#####
def find_csv(substring: str) -> Path:
    matches = list(DATA_DIR.glob(f'*{substring}*.csv'))
    if not matches:
        raise FileNotFoundError(f"No file matching '{substring}' in {DATA_DIR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple files match '{substring}': {matches}")
    return matches[0]

#####
# 1b) TRY TO LOAD A CSV, BUT SKIP IF MISSING OR AMBIGUOUS
#####
def try_load(substring: str) -> pd.DataFrame | None:
    try:
        path = find_csv(substring)
    except (FileNotFoundError, ValueError) as e:
        print(f"Skipping '{substring}': {e}")
        return None
    print(f"Loaded '{substring}' from {path.name}")
    return pd.read_csv(path, dtype=str)

#####
# 2) LOAD ALL WELLBEING CSVS (skipping any that aren’t there yet)
#####
df_s1_pre     = try_load('pre_session_1')
df_s2to4_pre  = try_load('pre_sessions_2-4')
df_s2to4_post = try_load('post_sessions_1-3')
df_s4_post    = try_load('post_session_4')
df_sms        = try_load('sms_day1,3,5')

#  — new step: trim SMS part_ids —
if df_sms is not None:
    df_sms['part_id'] = (
        df_sms['part_id']
              .astype(str)
              .str.strip()
              .str[:5]
    )
    print("Trimmed df_sms.part_id to first 5 characters")

df_final      = try_load('sms_post')

# Collect only the ones that actually loaded
all_dfs = [df for df in (
    df_s1_pre,
    df_s2to4_pre,
    df_s2to4_post,
    df_s4_post,
    df_sms,
    df_final
) if df is not None]

if not all_dfs:
    raise RuntimeError("No data files found to process!")

#####
# 3) CLEAN BDI ITEMS (extract leading number)
#####
for df in all_dfs:
    bdi_raw = [c for c in df.columns if re.fullmatch(r"bdi_\d+(?:_1)?", c)]
    for col in bdi_raw:
        df[col] = (
            df[col].astype(str)
                  .str.extract(r'^(\d+)')[0]
                  .astype(float)
        )

#####
# 4) SUM SCALES
#####
def sum_scale(df, cols, new_col):
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
    df[new_col] = df[cols].sum(axis=1)

def get_scale_cols(df, prefix, count, allow_suffix=False):
    cols = []
    for i in range(1, count+1):
        main = f"{prefix}_{i}"
        alt  = f"{prefix}_{i}_1"
        if main in df.columns:
            cols.append(main)
        elif allow_suffix and alt in df.columns:
            cols.append(alt)
    return cols

for df in all_dfs:
    # BAI
    bai_cols = get_scale_cols(df, 'bai', 21)
    if bai_cols:
        sum_scale(df, bai_cols, 'bai_sum')
    # BDI
    bdi_cols = get_scale_cols(df, 'bdi', 21)
    if bdi_cols:
        sum_scale(df, bdi_cols, 'bdi_sum')
    # MADRS
    madrs_cols = get_scale_cols(df, 'madrs', 9, allow_suffix=True)
    if madrs_cols:
        sum_scale(df, madrs_cols, 'MADRS_S_sum')

#####
# 5) STANDARDIZE M3VAS
#####
def standardize_m3vas(df):
    to_drop = []
    for var in ('suicidal','mood','pleasure'):
        # find change‐score first, then raw‐score
        ch_col  = next((c for c in df.columns
                        if re.fullmatch(rf"m3vas_ch_{var}(?:_1)?", c)), None)
        raw_col = next((c for c in df.columns
                        if re.fullmatch(rf"m3vas_{var}(?:_1)?",    c)), None)

        if ch_col:
            vals = pd.to_numeric(df[ch_col], errors='coerce') - 50
            to_drop.append(ch_col)
        elif raw_col:
            vals = pd.to_numeric(df[raw_col], errors='coerce') - 50
            to_drop.append(raw_col)
        else:
            continue

        if var == 'suicidal':
            # leave as-is: + means more suicidal (worse), so improvement → down
            df[f"m3vas_{var}"] = vals
        else:
            # mood & pleasure: + means improvement, so flip sign
            df[f"m3vas_{var}"] = -vals

    # drop only the original raw/change columns
    df.drop(columns=to_drop, inplace=True, errors='ignore')

# apply to each dataframe
for df in all_dfs:
    standardize_m3vas(df)

#####
# 6) CLEAN FISBER
#####
for df in all_dfs:
    for col in ('fisber_1','fisber_2','fisber_3'):
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                       .str.extract(r'(\d+)$')[0]
                       .astype(float)
            )

#####
# 7) COERCE TYPES
#####
# session_n for sessions
for df in (df_s1_pre, df_s2to4_pre, df_s2to4_post, df_s4_post):
    if df is not None and 'session_n' in df.columns:
        df['session_n'] = pd.to_numeric(df['session_n'], errors='coerce').astype('Int64')

# session_n & sms_n for SMS
if df_sms is not None:
    for col in ('session_n','sms_n'):
        if col in df_sms.columns:
            df_sms[col] = pd.to_numeric(df_sms[col], errors='coerce').astype('Int64')

# numeric scales
numeric_cols = [
    'phq9_sum','spane_p','spane_n',
    'm3vas_mood','m3vas_pleasure','m3vas_suicidal',
    'fisber_1','fisber_2','fisber_3',
    'bai_sum','bdi_sum','MADRS_S_sum'
]
for df in all_dfs:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

#####
# 8) ENSURE EVERY DF HAS part_id
#####
for df in all_dfs:
    if 'part_id' not in df.columns:
        if 'ProlificID' in df.columns:
            df.rename(columns={'ProlificID':'part_id'}, inplace=True)
        else:
            df['part_id'] = df.index.astype(str)
    df['part_id'] = df['part_id'].astype(str).str.strip()

#####
# 9) TAG TIMEPOINTS
#####
if df_s1_pre    is not None: df_s1_pre    ['timepoint'] = 'session1_pre'
if df_s2to4_pre is not None: df_s2to4_pre ['timepoint'] = df_s2to4_pre.apply(lambda r: f"session{r.session_n}_pre",  axis=1)
if df_s2to4_post is not None: df_s2to4_post['timepoint'] = df_s2to4_post.apply(lambda r: f"session{r.session_n}_post", axis=1)
if df_s4_post   is not None: df_s4_post   ['timepoint'] = 'session4_post'
if df_sms       is not None: df_sms       ['timepoint'] = df_sms.apply(lambda r: f"SMS_{r.session_n}_{r.sms_n}", axis=1)
if df_final     is not None: df_final     ['timepoint'] = 'final_post'

#####
# 10) CONCATENATE
#####
df_long = pd.concat(all_dfs, ignore_index=True)

#####
# 11) DROP NON-NUMERIC part_id
#####
df_long = df_long[df_long['part_id'].str.match(r'^\d+$', na=False)]

#####
# 12) MERGE IN CONDITION (if assignments file was found)
#####
df_assign = try_load('wp2_assignments')
if df_assign is not None:
    df_long = df_long.merge(
        df_assign[['part_id','condition']],
        on='part_id',
        how='inner'
    )
else:
    print("No assignments file: skipping merge with 'condition' column")

#####
# 13) DEFINE & APPLY ORDERED CATEGORICAL FOR timepoint
#####
ordered_timepoints = [
    'session1_pre','session1_post',
    'SMS_1_1','SMS_1_2','SMS_1_3',
    'session2_pre','session2_post',
    'SMS_2_1','SMS_2_2','SMS_2_3',
    'session3_pre','session3_post',
    'SMS_3_1','SMS_3_2','SMS_3_3',
    'session4_pre','session4_post',
    'final_post'
]
df_long['timepoint'] = pd.Categorical(
    df_long['timepoint'],
    categories=ordered_timepoints,
    ordered=True
)

#####
# 14) PLOTTING + FLAGGING ACROSS ALL PARTICIPANTS (BLINDED)
#####
bad_if_higher = {
    'phq9_sum',
    'spane_n',
    'fisber_1','fisber_2','fisber_3',
    'bai_sum','bdi_sum','MADRS_S_sum',
    'm3vas_suicidal'   # higher = more suicidal = worse
}

bad_if_lower = {
    'spane_p',
    'm3vas_mood',      # higher = better mood, so a drop is worsening
    'm3vas_pleasure'   # higher = better pleasure, so a drop is worsening
}

mcid_thresholds = {
    'phq9_sum': 5,
    'spane_p': 2,
    'spane_n': 2,
    'm3vas_mood': 12,
    'm3vas_pleasure': 12,
    'm3vas_suicidal': 12
}

#####
# ADJUST M3VAS SO 50 → 0 (“no change”), 0 → –50 (worse), 100 → +50 (better)
#####
def standardize_m3vas(df):
    for var in ('suicidal','mood','pleasure'):
        # detect the two possible inputs
        raw_col = next((c for c in df.columns
                        if re.fullmatch(f"m3vas_{var}(?:_1)?", c)), None)
        ch_col  = next((c for c in df.columns
                        if re.fullmatch(f"m3vas_ch_{var}(?:_1)?", c)),  None)

        if raw_col is not None:
            # pre‐session raw VAS: 0 (best) → 100 (worst)
            vals = pd.to_numeric(df[raw_col], errors='coerce') - 50
        elif ch_col is not None:
            # post‐session change VAS: already –50 (better) → +50 (worse)
            vals = pd.to_numeric(df[ch_col], errors='coerce')
        else:
            continue

        # for mood & pleasure, invert so higher = more distress
        if var in ('mood','pleasure'):
            vals = -vals

        df[f"m3vas_{var}"] = vals

    # drop all the old raw / ch columns
    drop_re = re.compile(
        r"^(?:m3vas_ch_(?:mood|pleasure|suicidal)(?:_1)?"
        r"|m3vas_(?:mood|pleasure|suicidal)_1)$"
    )
    to_drop = [c for c in df.columns if drop_re.match(c)]
    df.drop(columns=to_drop, errors='ignore', inplace=True)

# Apply to each dataframe
for df in all_dfs:
    standardize_m3vas(df)

#####
# PLOTTING & FLAGGING FUNCTION
#####
def plot_trajectories_and_flag(measure):
    if measure not in df_long.columns:
        print(f"→ skipping '{measure}': not in df_long")
        return

    # 1) Copy & drop session*_post for PHQ-9 and SPANE scales
    df_plot = df_long.copy()
    if measure in ('phq9_sum','spane_p','spane_n'):
        mask = df_plot['timepoint'].astype(str).str.match(r'^session\d+_post$')
        df_plot = df_plot[~mask]

    # 2) Determine which timepoints remain, preserving order
    timepoints_present = [tp for tp in ordered_timepoints if tp in df_plot['timepoint'].values]
    code_map = {tp: i for i, tp in enumerate(ordered_timepoints)}

    # 3) Plot trajectories
    fig, ax = plt.subplots(figsize=(12,6))
    for pid, grp in df_plot.groupby('part_id'):
        s = grp.dropna(subset=['timepoint', measure]).sort_values('timepoint')
        if s.empty:
            continue
        x = s['timepoint'].cat.codes
        y = s[measure].astype(float)
        ax.plot(x, y, linewidth=1, alpha=0.4)
        ax.text(x.iloc[-1] + 0.2, y.iloc[-1], pid, fontsize=5, va='center')

    # 4) Set ticks only for the retained timepoints
    tick_locs   = [code_map[tp] for tp in timepoints_present]
    tick_labels = timepoints_present
    ax.set_xticks(tick_locs)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right')

    ax.set_ylabel(measure)
    ax.set_title(f"{measure} — all participants")
    plt.tight_layout()
    plt.show()

    # 5) Flag first MCID‐level worsening events
    if measure in mcid_thresholds:
        th = mcid_thresholds[measure]
        breach_events = []

        for pid, grp_pid in df_plot.groupby('part_id'):
            seq = (
                grp_pid
                .dropna(subset=['timepoint', measure])
                .sort_values('timepoint')
                .loc[:, ['part_id','timepoint', measure]]
                .assign(change=lambda d: d[measure] - d[measure].iloc[0])
            )
            if measure in bad_if_higher:
                mask = seq['change'] >= th
            else:
                mask = seq['change'] <= -th

            if mask.any():
                breach_events.append(seq[mask].iloc[0])

        if breach_events:
            df_breaches = pd.DataFrame(breach_events)
            print(f"First MCID‐level worsening events (threshold ≥ {th}):")
            display(df_breaches)
        else:
            print(f"No participants worsened by at least the MCID of {th} for {measure}.")

    else:
        # 6) Fallback: any participant whose final point is worse than their first
        worsened = []
        for pid, grp_pid in df_plot.groupby('part_id'):
            seq = grp_pid.dropna(subset=['timepoint', measure]) \
                         .sort_values('timepoint')[measure] \
                         .astype(float)
            if len(seq) < 2:
                continue
            if ((measure in bad_if_higher and seq.iloc[-1] > seq.iloc[0]) or
                (measure in bad_if_lower  and seq.iloc[-1] < seq.iloc[0])):
                worsened.append(pid)

        if worsened:
            print("Worsened trajectories (blinded):")
            display(
                df_plot[df_plot['part_id'].isin(worsened)]
                       [['part_id','timepoint',measure]]
                       .dropna(subset=[measure])
                       .sort_values(['part_id','timepoint'])
                       .reset_index(drop=True)
            )
        else:
            print("No worsening detected across the sample.")

#####
# RUN FOR ALL MEASURES
#####
for m in measures:
    plot_trajectories_and_flag(m)